# EXP-25: Label Cleaning (Conservative) — Remove Mislabeled Samples

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** V12 pipeline (0.804) — 3 SVCs + 1 LGB  
**Enhancement:** OOF-based label cleaning — remove amostras com label inconsistente

**Approach:**
1. Rodar motor V12 com GroupKFold 5-fold → OOF probabilities
2. Identificar amostras onde OOF confidence no label original é muito baixa
   E todos os modelos concordam numa classe diferente
3. REMOVER essas amostras do treino (approach conservador)
4. Retreinar V12 nos dados limpos
5. Threshold optimization + submission

**Insight:** O organizador confirmou que label cleaning é permitido.
Uma competidora (53rd) confirmou que o dataset tem inconsistências.
**Runtime:** CPU only

In [ ]:
import os, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train_orig = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7

print(f'Train: {train_orig.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train_orig["target"].value_counts().sort_index()}')

In [ ]:
def clean_achados(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados:(.*?)(análise comparativa:|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def stable_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def extract_dense_features(df):
    features = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    features['report_length']   = text_col.apply(len)
    features['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    features['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    features['has_distortion']  = text_col.str.contains(r'distorção arquitetural', regex=True).astype(int)
    features['has_biopsy']      = text_col.str.contains(r'biopsy|biópsia|resultado de cine|carcinoma', regex=True).astype(int)
    return csr_matrix(features.values)

def prepare_features(train_df, test_df):
    """Build all TF-IDF + dense features from scratch."""
    for df in [train_df, test_df]:
        df['achados'] = df['report'].apply(clean_achados)
        df['full']    = df['report'].apply(clean_full)
    
    tfidf_A = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                                 sublinear_tf=True, max_features=80000))
    ])
    X_train_A = tfidf_A.fit_transform(train_df["achados"].values)
    X_test_A  = tfidf_A.transform(test_df["achados"].values)
    
    tfidf_F = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                                 sublinear_tf=True, max_features=80000))
    ])
    X_train_F = tfidf_F.fit_transform(train_df["full"].values)
    X_test_F  = tfidf_F.transform(test_df["full"].values)
    
    tfidf_F2 = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                                 sublinear_tf=True, max_features=100000))
    ])
    X_train_F2 = tfidf_F2.fit_transform(train_df["full"].values)
    X_test_F2  = tfidf_F2.transform(test_df["full"].values)
    
    X_train_dense = extract_dense_features(train_df)
    X_test_dense  = extract_dense_features(test_df)
    
    X_train_lgb = hstack([X_train_A, X_train_dense]).tocsr()
    X_test_lgb  = hstack([X_test_A,  X_test_dense]).tocsr()
    
    return {
        'A': (X_train_A, X_test_A),
        'F': (X_train_F, X_test_F),
        'F2': (X_train_F2, X_test_F2),
        'lgb': (X_train_lgb, X_test_lgb),
    }

print("Functions defined.")

In [ ]:
# =============================================================================
# Phase 1: Train V12 on ORIGINAL data to get OOF predictions for label analysis
# =============================================================================
print("Phase 1: Training V12 on original data for label analysis...")

train = train_orig.copy()
train['group'] = train['report'].apply(stable_hash)
feats = prepare_features(train, test)
y = train['target'].astype(int).values
groups = train['group'].values

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(feats['A'][0], y, groups))

# Train 4 models, collect OOF probabilities for each
model_oof = {}
for name, X_key in [('svc_A', 'A'), ('svc_F', 'F'), ('svc_F2', 'F2')]:
    X_tr = feats[X_key][0]
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = CalibratedClassifierCV(
            LinearSVC(class_weight="balanced", random_state=42, max_iter=10000),
            cv=3, method='sigmoid'
        )
        m.fit(X_tr[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X_tr[va_idx])
    model_oof[name] = oof
    print(f"  {name} OOF F1: {f1_score(y, np.argmax(oof, axis=1), average='macro'):.4f}")

X_lgb = feats['lgb'][0]
oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = lgb.LGBMClassifier(class_weight='balanced', n_estimators=300, learning_rate=0.05,
                           max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_lgb[tr_idx], y[tr_idx])
    oof[va_idx] = m.predict_proba(X_lgb[va_idx])
model_oof['lgb'] = oof
print(f"  lgb OOF F1: {f1_score(y, np.argmax(oof, axis=1), average='macro'):.4f}")

# V12 blend
oof_blend = 0.70 * (0.25 * model_oof['svc_A'] + 0.40 * model_oof['svc_F'] + 0.35 * model_oof['svc_F2']) + 0.30 * model_oof['lgb']
original_f1 = f1_score(y, np.argmax(oof_blend, axis=1), average='macro')
print(f"\nOriginal V12 OOF F1: {original_f1:.4f}")

In [ ]:
# =============================================================================
# Phase 2: Identify mislabeled samples
# Criteria: OOF probability of the assigned label is very low
# AND all individual models agree on a different class
# =============================================================================

oof_label_prob = oof_blend[np.arange(len(y)), y]  # P(assigned label) per sample

# Per-model predicted class
model_preds = {}
for name, oof in model_oof.items():
    model_preds[name] = np.argmax(oof, axis=1)

# Find samples where ALL models disagree with the label
all_disagree = np.ones(len(y), dtype=bool)
for name, preds in model_preds.items():
    all_disagree &= (preds != y)

# Additionally, all models agree on the SAME alternative class
alt_classes = np.stack([model_preds[name] for name in model_preds], axis=1)
all_agree_alt = np.all(alt_classes == alt_classes[:, 0:1], axis=1)

# Conservative: remove only if P(label) < 0.10 AND all models disagree AND all agree on same alt
CONFIDENCE_THRESHOLD = 0.10
suspect_mask = (oof_label_prob < CONFIDENCE_THRESHOLD) & all_disagree & all_agree_alt

n_suspect = suspect_mask.sum()
print(f"Suspect samples (P(label)<{CONFIDENCE_THRESHOLD}, all models disagree, all agree on alt):")
print(f"  Total: {n_suspect} / {len(y)} ({n_suspect/len(y)*100:.2f}%)")
print(f"\nSuspect samples by original class:")
suspect_df = train[suspect_mask].copy()
suspect_df['oof_pred'] = np.argmax(oof_blend[suspect_mask], axis=1)
suspect_df['oof_prob'] = oof_label_prob[suspect_mask]
print(suspect_df.groupby('target')['oof_prob'].agg(['count', 'mean']).to_string())

print(f"\nSuspect label → predicted:")
if n_suspect > 0:
    for _, row in suspect_df.head(20).iterrows():
        print(f"  ID={row['ID']}: label={row['target']} → pred={row['oof_pred']} (P(label)={row['oof_prob']:.4f})")

In [ ]:
# =============================================================================
# Phase 3: Retrain V12 on CLEANED data (suspect samples removed)
# =============================================================================
print(f"\nPhase 3: Retraining V12 on cleaned data ({len(train) - n_suspect} samples)...")

train_clean = train[~suspect_mask].reset_index(drop=True)
print(f"Removed {n_suspect} samples. Clean train: {train_clean.shape}")
print(f"Clean class distribution:\n{train_clean['target'].value_counts().sort_index()}")

# Rebuild features on clean data
feats_clean = prepare_features(train_clean, test)
y_clean = train_clean['target'].astype(int).values
groups_clean = train_clean['report'].apply(stable_hash).values

gkf_clean = GroupKFold(n_splits=N_FOLDS)
folds_clean = list(gkf_clean.split(feats_clean['A'][0], y_clean, groups_clean))

# Train V12 on clean data with CV
clean_oof = {}
clean_test_probas = {}

for name, X_key in [('svc_A', 'A'), ('svc_F', 'F'), ('svc_F2', 'F2')]:
    X_tr, X_te = feats_clean[X_key]
    oof = np.zeros((len(train_clean), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds_clean):
        m = CalibratedClassifierCV(
            LinearSVC(class_weight="balanced", random_state=42, max_iter=10000),
            cv=3, method='sigmoid'
        )
        m.fit(X_tr[tr_idx], y_clean[tr_idx])
        oof[va_idx] = m.predict_proba(X_tr[va_idx])
        te_acc += m.predict_proba(X_te) / N_FOLDS
    clean_oof[name] = oof
    clean_test_probas[name] = te_acc
    print(f"  {name} clean OOF F1: {f1_score(y_clean, np.argmax(oof, axis=1), average='macro'):.4f}")

X_lgb_tr, X_lgb_te = feats_clean['lgb']
oof = np.zeros((len(train_clean), N_CLASSES), dtype=np.float64)
te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds_clean):
    m = lgb.LGBMClassifier(class_weight='balanced', n_estimators=300, learning_rate=0.05,
                           max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X_lgb_tr[tr_idx], y_clean[tr_idx])
    oof[va_idx] = m.predict_proba(X_lgb_tr[va_idx])
    te_acc += m.predict_proba(X_lgb_te) / N_FOLDS
clean_oof['lgb'] = oof
clean_test_probas['lgb'] = te_acc
print(f"  lgb clean OOF F1: {f1_score(y_clean, np.argmax(oof, axis=1), average='macro'):.4f}")

# V12 blend on clean data
oof_clean_blend = 0.70 * (0.25 * clean_oof['svc_A'] + 0.40 * clean_oof['svc_F'] + 0.35 * clean_oof['svc_F2']) + 0.30 * clean_oof['lgb']
te_clean_blend = 0.70 * (0.25 * clean_test_probas['svc_A'] + 0.40 * clean_test_probas['svc_F'] + 0.35 * clean_test_probas['svc_F2']) + 0.30 * clean_test_probas['lgb']

clean_f1 = f1_score(y_clean, np.argmax(oof_clean_blend, axis=1), average='macro')
print(f"\nClean V12 OOF F1: {clean_f1:.4f} (original: {original_f1:.4f}, delta: {clean_f1 - original_f1:+.4f})")

In [ ]:
# =============================================================================
# Phase 4: Threshold optimization on clean OOF
# =============================================================================

baseline_preds = np.argmax(oof_clean_blend, axis=1)
baseline_f1 = f1_score(y_clean, baseline_preds, average='macro')

threshold_classes = [6, 5, 4, 3]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in [6, 5, 4, 3]:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in [c for c in [6, 5, 4, 3] if c != cls]:
                if higher_cls in thresholds and threshold_classes.index(higher_cls) < threshold_classes.index(cls):
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

best_thresholds = {}
current_f1 = baseline_f1

for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_clean_blend, trial)
        trial_f1 = f1_score(y_clean, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f"Class {cls}: threshold={best_cls_thr:.2f}, macro-F1={best_cls_f1:.4f}")
    else:
        print(f"Class {cls}: no improvement")

final_f1 = f1_score(y_clean, apply_thresholds(oof_clean_blend, best_thresholds), average='macro')
print(f"\nFinal clean OOF F1 with thresholds: {final_f1:.4f}")
print(f"Thresholds: {best_thresholds}")

In [ ]:
# =============================================================================
# Submission
# =============================================================================

test_preds = apply_thresholds(te_clean_blend, best_thresholds)

def apply_safe_rules(row):
    text = str(row['report']).lower()
    current_pred = int(row['target'])
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b', text):
        return 6
    if 'espiculad' in text and 'distorção' in text and current_pred < 4:
        return 5
    return current_pred

test['target'] = test_preds
test['target'] = test.apply(apply_safe_rules, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)